# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset info
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

> **Note:** In `mlcroissant`, you can access the `record_sets` attribute of the `dataset.metadata` object to list the available record sets and their `@id`. We then show the fields and columns associated with each record set.

In [ ]:
# List all record sets and their schema field IDs
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field @id: {field.id} ({getattr(field, 'name', 'N/A')}) [dataType: {getattr(field, 'data_type', 'N/A')}]" )
    print("")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.

In this dataset, there is a primary record set containing patient clinicopathological data. We'll extract records from that record set, using its `@id`, and load them into a DataFrame for further analysis.

> **Note:** Always use the record set's `@id` for all references.

In [ ]:
# Identify record set @id(s) for data extraction
record_sets = [rs.id for rs in metadata.record_sets]
print("Available record set @id(s):", record_sets)

# Prepare DataFrame(s) for analysis
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns of main record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"Columns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Here, we will select numeric and categorical fields (by their field `@id`), filter and normalize their values, and group data as an example. 

We refer to fields such as patient age, diagnostic intervals, or other relevant numeric fields using their Croissant field `@id`, ensuring transparent and reproducible analyses.

In [ ]:
# Example EDA: pick a numeric field and a grouping field from the schema overview
main_df = dataframes[main_record_set_id]
# Replace these with the actual @id values for numeric and categorical fields, as found above
# For illustration, let's assume following field @ids (please replace with actual @ids from the overview):
numeric_field_id = None
group_field_id = None
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        for field in getattr(rs, 'fields', []):
            if hasattr(field, 'data_type') and field.data_type in ["schema:Integer", "schema:Number", "schema:Float"] and numeric_field_id is None:
                numeric_field_id = field.id
            if hasattr(field, 'data_type') and field.data_type in ["schema:Text", "schema:DefinedTermSet"] and group_field_id is None:
                group_field_id = field.id
            if numeric_field_id and group_field_id:
                break
        break

print(f"Numeric field selected for filtering: {numeric_field_id}")
print(f"Group-by field selected: {group_field_id}")

# EDA only if such fields are found
if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0

    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records from {main_record_set_id} with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group-by operation
    if group_field_id and group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions—for example, a histogram of the numeric field, or a boxplot by group.

Remember to use the DataFrame columns named by their field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field, if present
if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a dataset defined by a Croissant schema using the `mlcroissant` library
- Explore record sets, fields, and their `@id`s
- Extract data into DataFrames using record set `@id`
- Perform basic exploratory data analysis (EDA) and visualization referencing fields by their Croissant `@id`

This ensures reproducibility and clarity when working with FAIRML Croissant-compliant datasets!